# Exploratory Data Analysis - MaIA Scoliosis Dataset

**Proyecto:** Segmentacion automatica de columna vertebral y vertebras en radiografias de pacientes sanos y con escoliosis

**Dataset:** 250 radiografias (71 Normal, 179 Escoliosis) con mascaras de segmentacion binaria y multiclase (36 clases).

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

from spine_segmentation.config import *
from spine_segmentation.data.class_mapping import get_class_names, remap_mask

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

print(f'Dataset root: {DATASET_ROOT}')
print(f'Dataset exists: {DATASET_ROOT.exists()}')

## 1. Dataset Overview

In [ ]:
# Load dataset index
df = pd.read_csv(DATASET_INDEX_CSV)
print(f'Total samples: {len(df)}')
print(f'\nCondition distribution:')
print(df['split'].value_counts())
print(f'\nColumns: {list(df.columns)}')
df.head()

In [ ]:
# Class distribution pie chart
fig, ax = plt.subplots(figsize=(8, 6))
counts = df['split'].value_counts()
ax.pie(counts.values, labels=counts.index, autopct='%1.1f%%',
       colors=['#2ecc71', '#e74c3c'], startangle=90, textprops={'fontsize': 14})
ax.set_title('Distribucion Normal vs Escoliosis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Image Dimensions Analysis

In [ ]:
# Analyze image sizes
heights, widths, conditions = [], [], []
for _, row in df.iterrows():
    img = cv2.imread(str(DATASET_ROOT / row['radiograph_path']))
    if img is not None:
        h, w = img.shape[:2]
        heights.append(h)
        widths.append(w)
        conditions.append(row['split'])

size_df = pd.DataFrame({'height': heights, 'width': widths, 'condition': conditions})
size_df['aspect_ratio'] = size_df['height'] / size_df['width']

print('Image Size Statistics:')
print(size_df.describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(size_df['width'], size_df['height'], 
               c=size_df['condition'].map({'Normal': 'green', 'Scoliosis': 'red'}),
               alpha=0.6, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('Width (px)', fontsize=12)
axes[0].set_ylabel('Height (px)', fontsize=12)
axes[0].set_title('Image Dimensions', fontsize=14)

sns.histplot(data=size_df, x='height', hue='condition', bins=20, ax=axes[1])
axes[1].set_title('Height Distribution', fontsize=14)

sns.histplot(data=size_df, x='aspect_ratio', hue='condition', bins=20, ax=axes[2])
axes[2].set_title('Aspect Ratio (H/W)', fontsize=14)

plt.suptitle('Image Dimension Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Multiclass Label Distribution

In [ ]:
# Analyze pixel-level class distribution
class_names = get_class_names('vertebrae_24')
class_pixel_counts = Counter()
class_image_counts = Counter()

for _, row in df.iterrows():
    mask_path = DATASET_ROOT / row['multiclass_id_png']
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
    if mask is None:
        continue
    if mask.ndim == 3:
        mask = mask[:, :, 0]
    # Remap to 24-class scheme
    mask = remap_mask(mask, 'vertebrae_24')
    for c in np.unique(mask):
        class_pixel_counts[c] += (mask == c).sum()
        class_image_counts[c] += 1

# Build summary table
total_pixels = sum(class_pixel_counts.values())
rows = []
for c in sorted(class_pixel_counts.keys()):
    rows.append({
        'Class ID': c,
        'Name': class_names.get(c, f'class_{c}'),
        'Pixel Count': class_pixel_counts[c],
        'Pixel %': class_pixel_counts[c] / total_pixels * 100,
        'Images': class_image_counts[c],
    })
class_df = pd.DataFrame(rows)
class_df

In [ ]:
# Plot vertebra pixel distribution (excluding background)
vert_df = class_df[(class_df['Class ID'] > 0) & (class_df['Class ID'] < 23)].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors = []
for _, row in vert_df.iterrows():
    name = row['Name']
    if name.startswith('C'):
        colors.append('#e74c3c')
    elif name.startswith('T'):
        colors.append('#2ecc71')
    elif name.startswith('L'):
        colors.append('#3498db')
    else:
        colors.append('#95a5a6')

axes[0].barh(vert_df['Name'], vert_df['Pixel Count'], color=colors)
axes[0].set_xlabel('Total Pixel Count', fontsize=12)
axes[0].set_title('Pixel Distribution per Vertebra', fontsize=14)

axes[1].barh(vert_df['Name'], vert_df['Images'], color=colors)
axes[1].set_xlabel('Number of Images', fontsize=12)
axes[1].set_title('Image Presence per Vertebra', fontsize=14)

plt.suptitle('Vertebra Class Distribution (24-class scheme)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Cobb Angle Distribution

In [ ]:
# Load Cobb angle ground truth
metrics_df = pd.read_csv(RADIOGRAPH_METRICS_CSV)
cobb = metrics_df['cobb_angle_deg']

print(f'Cobb Angle Statistics (n={len(cobb)}):')
print(f'  Min:    {cobb.min():.1f} deg')
print(f'  Max:    {cobb.max():.1f} deg')
print(f'  Mean:   {cobb.mean():.1f} deg')
print(f'  Median: {cobb.median():.1f} deg')
print(f'  Std:    {cobb.std():.1f} deg')
print(f'\nSeverity:')
print(f'  Mild   (10-25 deg): {((cobb >= 10) & (cobb < 25)).sum()}')
print(f'  Moderate (25-40):   {((cobb >= 25) & (cobb < 40)).sum()}')
print(f'  Severe   (>40):     {(cobb >= 40).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cobb, bins=25, color='mediumpurple', edgecolor='white', alpha=0.8)
axes[0].axvline(x=10, color='green', linestyle='--', linewidth=2, label='10 deg')
axes[0].axvline(x=25, color='orange', linestyle='--', linewidth=2, label='25 deg')
axes[0].axvline(x=40, color='red', linestyle='--', linewidth=2, label='40 deg')
axes[0].set_xlabel('Cobb Angle (degrees)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Cobb Angle Distribution', fontsize=14)
axes[0].legend(fontsize=10)

sns.boxplot(data=cobb, ax=axes[1], color='mediumpurple')
axes[1].set_ylabel('Cobb Angle (degrees)', fontsize=12)
axes[1].set_title('Cobb Angle Box Plot', fontsize=14)

plt.suptitle('Cobb Angle Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Sample Visualizations

In [ ]:
from spine_segmentation.config import MULTICLASS_COLORS

# Show sample radiographs with their multiclass segmentation overlays
sample_normal = df[df['split'] == 'Normal'].sample(2, random_state=42)
sample_scol = df[df['split'] == 'Scoliosis'].sample(4, random_state=42)
samples = pd.concat([sample_normal, sample_scol]).reset_index(drop=True)

fig, axes = plt.subplots(2, 6, figsize=(24, 12))

for i, (_, row) in enumerate(samples.iterrows()):
    # Original image
    img = cv2.imread(str(DATASET_ROOT / row['radiograph_path']))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"{row['image']}\n({row['split']})", fontsize=10)
    axes[0, i].axis('off')
    
    # Multiclass overlay
    mask = cv2.imread(str(DATASET_ROOT / row['multiclass_id_png']), cv2.IMREAD_UNCHANGED)
    if mask is not None:
        if mask.ndim == 3:
            mask = mask[:, :, 0]
        overlay = img.copy()
        color_mask = np.zeros_like(img)
        for cid, color in MULTICLASS_COLORS.items():
            if cid == 0:
                continue
            region = mask == cid
            if region.any():
                color_mask[region] = color
        overlay = cv2.addWeighted(overlay, 0.6, color_mask, 0.4, 0)
        axes[1, i].imshow(overlay)
    axes[1, i].set_title('Vertebrae Segmentation', fontsize=10)
    axes[1, i].axis('off')

plt.suptitle('Sample Radiographs: Original (top) vs Multiclass Overlay (bottom)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Data Split Verification

In [ ]:
from spine_segmentation.data.splits import load_splits

splits = load_splits()

for split_name, images in splits.items():
    split_df = df[df['image'].isin(images)]
    counts = split_df['split'].value_counts()
    total = len(images)
    print(f'{split_name:>6}: {total:3d} images | '
          f'Normal={counts.get("Normal", 0):2d} ({counts.get("Normal", 0)/total*100:.0f}%) | '
          f'Scoliosis={counts.get("Scoliosis", 0):3d} ({counts.get("Scoliosis", 0)/total*100:.0f}%)')

---
## Summary

- **250 radiographs**: 71 Normal (28.4%), 179 Scoliosis (71.6%)
- **Variable image sizes**: portrait-oriented, aspect ratios ~2.5-3.5
- **22 vertebra classes** + background + other structures = 24 classes (scheme used)
- **Class imbalance**: background dominates; cervical vertebrae have fewer pixels than lumbar
- **Cobb angles**: range from ~15 to ~90 degrees, majority moderate-to-severe
- **Data split**: 70/15/15 stratified by condition